In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import json
import sys

model_paths = [
    "/kaggle/input/models/leonidtikhanov/pinn-model/pytorch/default/7",
    "",
]

for p in model_paths:
    if Path(p).exists():
        sys.path.insert(0, p)
        break

import pinn_model
pinn_model = importlib.reload(pinn_model)

print(pinn_model.__file__)
print("run_experiment:", hasattr(pinn_model, "run_experiment"))


In [ ]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = "cuda"
    print("gpu:", torch.cuda.get_device_name(0))
else:
    device = "cpu"

work_dir = Path("/kaggle/working")
if not work_dir.exists():
    work_dir = Path(".")

out_dir = work_dir / "results_exp_13_helmholtz1d_width_lbfgs_resample"
out_dir.mkdir(parents=True, exist_ok=True)

print("device:", device)
print("work_dir:", work_dir)
print("out_dir:", out_dir)


In [ ]:
m_values = [10, 12, 14]
dtype_values = ["fp32", "fp64"]
seed_values = [0]

variants = [
    {
        "variant": "width128_adam5000_lbfgs3000_inner5_resample250",
        "adam_steps": 5000,
        "lbfgs_steps": 3000,
        "lr_adam": 5e-4,
        "lbfgs_max_iter": 5,
        "lbfgs_max_eval": None,
        "lbfgs_lr": 1.0,
        "lbfgs_history_size": 100,
        "lbfgs_tolerance_grad": 1e-10,
        "lbfgs_tolerance_change": 1e-12,
        "lbfgs_line_search_fn": "strong_wolfe",
        "resample_every": 250,
        "point_mode": None,
        "n_collocation": 8000,
        "hid_size": 128,
        "num_layers": 4,
        "init_gain": None,
        "bc_weight": 1.0,
        "hard_bc": False,
    },
    {
        "variant": "width192_adam5000_lbfgs3000_inner5_resample250",
        "adam_steps": 5000,
        "lbfgs_steps": 3000,
        "lr_adam": 5e-4,
        "lbfgs_max_iter": 5,
        "lbfgs_max_eval": None,
        "lbfgs_lr": 1.0,
        "lbfgs_history_size": 100,
        "lbfgs_tolerance_grad": 1e-10,
        "lbfgs_tolerance_change": 1e-12,
        "lbfgs_line_search_fn": "strong_wolfe",
        "resample_every": 250,
        "point_mode": None,
        "n_collocation": 8000,
        "hid_size": 192,
        "num_layers": 4,
        "init_gain": None,
        "bc_weight": 1.0,
        "hard_bc": False,
    },
]

runs = []
i = 1
for m in m_values:
    for variant in variants:
        for dtype in dtype_values:
            for seed in seed_values:
                run = variant.copy()
                run["run_id"] = i
                run["m"] = m
                run["dtype"] = dtype
                run["seed"] = seed
                runs.append(run)
                i += 1

pd.DataFrame(runs)[["run_id", "m", "variant", "dtype", "seed", "hid_size", "adam_steps", "lbfgs_steps", "lbfgs_max_iter", "n_collocation"]]


In [ ]:
base_config = {
    "task_name": "helmholtz1d",
    "dtype": "fp32",
    "seed": 0,
    "device": device,
    "m": m_values[0],
    "lambda_val": 1.0,
    "hid_size": 128,
    "num_layers": 4,
    "init_gain": None,
    "n_collocation": 8000,
    "point_mode": None,
    "n_bc": 2,
    "adam_steps": 5000,
    "lbfgs_steps": 3000,
    "lr_adam": 5e-4,
    "resample_every": 250,
    "use_adam": True,
    "use_lbfgs": True,
    "lbfgs_tolerance_grad": 1e-10,
    "lbfgs_tolerance_change": 1e-12,
    "lbfgs_history_size": 100,
    "lbfgs_lr": 1.0,
    "lbfgs_max_iter": 5,
    "lbfgs_max_eval": None,
    "lbfgs_line_search_fn": "strong_wolfe",
    "hard_bc": False,
    "bc_weight": 1.0,
    "log_dir": str(out_dir / "runs" / "helmholtz_tmp"),
}

len(runs)


In [ ]:
all_summaries = []
all_histories = {}

for run in runs:
    config = base_config.copy()
    config.update(run)
    config["use_adam"] = config["adam_steps"] > 0
    config["use_lbfgs"] = config["lbfgs_steps"] > 0
    name = f"exp13_r{run['run_id']:03d}_m{run['m']}_{run['variant']}_{run['dtype']}_s{run['seed']}"
    config["log_dir"] = str(out_dir / "runs" / name)

    run_dir = Path(config["log_dir"])
    summary_file = run_dir / "summary.json"
    metrics_file = run_dir / "metrics.csv"

    if summary_file.exists() and metrics_file.exists():
        with open(summary_file) as f:
            summary = json.load(f)
        history = pd.read_csv(metrics_file)
    else:
        history, summary = pinn_model.run_experiment(config)

    summary["run_id"] = run["run_id"]
    summary["run_name"] = name
    summary["variant"] = run["variant"]
    summary["m"] = run["m"]
    summary["dtype"] = run["dtype"]
    summary["seed"] = run["seed"]
    summary["best_l2_error"] = float(history["l2_error"].min())
    summary["final_l2_error"] = float(history["l2_error"].iloc[-1])
    summary["elapsed_time"] = float(history["time_sec"].iloc[-1])
    all_summaries.append(summary)
    all_histories[name] = history

    print(run["run_id"], run["variant"], "m", run["m"], run["dtype"], "seed", run["seed"], "final", summary["final_l2_error"], "best", summary["best_l2_error"], "time", summary["elapsed_time"])

summary_df = pd.DataFrame(all_summaries)
summary_path = out_dir / "exp_13_helmholtz1d_width_lbfgs_resample_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_path


In [ ]:
cols = ["run_id", "variant", "m", "dtype", "seed", "best_l2_error", "final_l2_error", "elapsed_time"]
summary_df[cols].sort_values(["m", "variant", "dtype", "seed"])


In [ ]:
grouped = summary_df.groupby(["variant", "m", "dtype"])[["best_l2_error", "final_l2_error", "elapsed_time"]].agg(["mean", "std", "min", "max"])
grouped.columns = ["_".join(c).strip("_") for c in grouped.columns]
grouped = grouped.reset_index()
grouped.to_csv(out_dir / "exp_13_helmholtz1d_width_lbfgs_resample_grouped.csv", index=False)

ratio = grouped.pivot_table(index=["variant", "m"], columns="dtype", values="best_l2_error_mean").reset_index()
if "fp32" in ratio.columns and "fp64" in ratio.columns:
    ratio["fp64_over_fp32_best"] = ratio["fp64"] / ratio["fp32"]
ratio.to_csv(out_dir / "exp_13_helmholtz1d_width_lbfgs_resample_fp64_ratio.csv", index=False)
ratio


In [ ]:
for variant in summary_df["variant"].unique():
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for name, hist in all_histories.items():
        row = summary_df[summary_df["run_name"] == name].iloc[0]
        if row["variant"] != variant:
            continue
        label = f"m={row['m']} {row['dtype']} s{row['seed']}"
        ax[0].plot(hist["step"], hist["total_loss"], label=label)
        ax[1].plot(hist["step"], hist["l2_error"], label=label)
    ax[0].set_yscale("log")
    ax[1].set_yscale("log")
    ax[0].set_xlabel("step")
    ax[1].set_xlabel("step")
    ax[0].set_ylabel("total_loss")
    ax[1].set_ylabel("l2_error")
    ax[0].grid(True)
    ax[1].grid(True)
    ax[0].legend(fontsize=8)
    ax[1].legend(fontsize=8)
    fig.suptitle(variant)
    fig.tight_layout()
    plt.show()


In [ ]:
from IPython.display import Image, display

for row in summary_df.sort_values(["m", "variant", "dtype", "seed"]).itertuples():
    p = Path(row.log_dir) / "solution_t1.png"
    if p.exists():
        print(row.run_name)
        display(Image(filename=str(p)))


In [ ]:
summary_df[cols].sort_values("best_l2_error", ascending=False).head(20)
